In [ ]:
import json
import pandas as pd
from datetime import datetime
from IPython.display import display, Markdown
from os import listdir as ls
import pycountry

from emu_renewal.inputs import DATA_PATH, END_VACC_THRESHOLD, TEXT_DATE_FORMAT, get_country_vacc_data, get_indicator_series_from_who_data
from emu_renewal.run import find_run_end_time
from emu_renewal.outputs import add_bool_row_to_table
from emu_renewal.selection import is_mostly_zeros, find_null_data, find_neg_data, has_repeats, has_outlier

In [ ]:
g_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "gmob" in i]
fb_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "fbmob" in i]
either_mob_avail = list(set(g_avail + fb_avail))

summary = pd.DataFrame(index=either_mob_avail)
add_bool_row_to_table(summary, g_avail, "Google avail.")
add_bool_row_to_table(summary, fb_avail, "FB avail.")
txt = "To select countries for inclusion in our analysis, we first identified all countries " \
    "for which either Google or Facebook mobility data were available. "

In [ ]:
# Gather data
case_data = {}
death_data = {}
for c in either_mob_avail:
    vacc_data = get_country_vacc_data(c)
    end_time = find_run_end_time(vacc_data, END_VACC_THRESHOLD, c)

    data = get_indicator_series_from_who_data("New_cases", c)
    case_data[c] = data[data.index < end_time]

    data = get_indicator_series_from_who_data("New_deaths", c)
    death_data[c] = data[data.index < end_time]

txt += "Next, we considered the quality of the data for our two main WHO indicators " \
    'which we required for inclusion in the analysis: "New_cases" and "New_deaths" ' \
    "from the start of data availability through to the end time of the analysis for each country. "

In [ ]:
# Select countries based on data characteristics
n_repeats = 6
outlier_threshold = 2.0

no_cases = find_null_data(case_data)
no_deaths = find_null_data(death_data)
mostly_zeroes = [c for c, d in case_data.items() if is_mostly_zeros(d)]
add_bool_row_to_table(summary, no_cases, "No cases")
add_bool_row_to_table(summary, no_deaths, "No deaths")
txt += "\n\nUsing these data, we excluded any countries for which either case or death data was absent " \
    "and also excluded any countries for which the majority of case count estimates were zero. "

neg_cases = find_neg_data(case_data)
neg_deaths = find_neg_data(death_data)
add_bool_row_to_table(summary, no_cases, "Neg. cases")
add_bool_row_to_table(summary, no_cases, "Neg. deaths")
txt += "We also excluded any countries for which any negative values were present in the available data. "

repeat_threshold = 1e-10
case_outliers = [c for c, d in case_data.items() if has_outlier(d, outlier_threshold)]
death_outliers = [c for c, d in death_data.items() if has_outlier(d, outlier_threshold)]
add_bool_row_to_table(summary, case_outliers, "Case outliers")
add_bool_row_to_table(summary, death_outliers, "Death outliers")
txt += "We further excluded any countries for which single marked outliers were present, " \
    f"which we defined as a single value that was more than {round(outlier_threshold)} " \
    "times greater than the next highest estimate available during the analysis period. "

start_time = datetime(2020, 4, 1)
case_repeats = [c for c, d in case_data.items() if has_repeats(d[d.index > start_time], n_repeats, repeat_threshold)]
death_repeats = [c for c, d in death_data.items() if has_repeats(d[d.index > start_time], n_repeats, repeat_threshold)]
add_bool_row_to_table(summary, case_repeats, "Rep-eated cases")
add_bool_row_to_table(summary, death_repeats, "Rep-eated deaths")
txt += "Last we excluded any countries for which several repeated values were present, " \
    "or the change from each value to the subsequent reported was identical for several values. " \
    f"We set the threshold number of repeated values or repeated changes for exclusion at {n_repeats} consecutive repeats " \
    f"and required that these repeated values occur after {start_time.strftime(TEXT_DATE_FORMAT)} because " \
    "these repeated values tended to be small and less significant for calibration prior to this date."
display(Markdown(txt))

In [ ]:
excluded = set(no_cases + no_deaths + neg_cases + neg_deaths + case_repeats + death_repeats + case_outliers + death_outliers + mostly_zeroes)
included = [c for c in either_mob_avail if c not in excluded]
add_bool_row_to_table(summary, included, "Inc-luded")
included.append("AUS")

In [ ]:
summary.index = summary.index.map(lambda iso3: pycountry.countries.lookup(iso3).name)
display(Markdown(summary.to_markdown()))

In [ ]:
json.dump(included, open(DATA_PATH / "config/included.json", "w"))